# 01 FabricNET Feature Extraction & Specimen-Disjoint Regression

## 1. Mathematical Objective & Methodological Isolation
This notebook establishes the learned feature-regression baseline for fabric thread density estimation (ends/cm and picks/cm) using structural texture features from the literature (GLCM, LBP, Gabor filter banks, Intensity Feature Vectors).

### Invariants:
1. **Specimen-Level Disjoint Splitting**: Splits occur strictly across unique `Image id` specimens (78 train / 26 val / 26 test). No augmented crops or patches from the same specimen ever cross partition boundaries.
2. **Framework & Determinism**: All models and random seeds are locked to `seed=42`.
3. **No Synthetic Leakage**: Evaluated strictly on the held-out test split of 26 real FabricNET specimens.

In [1]:
# The mechanistic justification for this step is: initialize scientific computing libraries, ensure reproducible pseudo-random states, and verify hardware execution context.
import os
import glob
import json
import time
import numpy as np
import pandas as pd
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy import stats
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


CUDA Available: True
Device Name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# The mechanistic justification for this step is: enforce strict specimen-level disjoint partitioning stratified by weave architecture to prevent data leakage.
params_path = "data/fabricnet/FabricNet_parameters.xlsx"
df_meta = pd.read_excel(params_path)
print("Metadata shape:", df_meta.shape)
print("Weave distribution:\n", df_meta['Texture'].value_counts())

# Stratified split: 60% train (78), 20% val (26), 20% test (26)
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_val_idx, test_idx = next(sss_test.split(df_meta, df_meta['Texture']))

df_train_val = df_meta.iloc[train_val_idx]
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED) # 0.25 of 0.80 = 0.20 of total
train_sub_idx, val_sub_idx = next(sss_val.split(df_train_val, df_train_val['Texture']))

train_ids = df_train_val.iloc[train_sub_idx]['Image id'].tolist()
val_ids = df_train_val.iloc[val_sub_idx]['Image id'].tolist()
test_ids = df_meta.iloc[test_idx]['Image id'].tolist()

print(f"Train IDs: {len(train_ids)} | Val IDs: {len(val_ids)} | Test IDs: {len(test_ids)}")

# Verify zero leakage across splits
assert len(set(train_ids).intersection(set(val_ids))) == 0, "Leakage between train and val"
assert len(set(train_ids).intersection(set(test_ids))) == 0, "Leakage between train and test"
assert len(set(val_ids).intersection(set(test_ids))) == 0, "Leakage between val and test"

splits = {
    "seed": SEED,
    "train_ids": sorted(train_ids),
    "val_ids": sorted(val_ids),
    "test_ids": sorted(test_ids),
    "train_count": len(train_ids),
    "val_count": len(val_ids),
    "test_count": len(test_ids)
}

os.makedirs("experiments/learned_models/results", exist_ok=True)
with open("experiments/learned_models/results/fabricnet_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

print("Splits successfully serialized.")


Metadata shape: (130, 5)
Weave distribution:
 Texture
plain    50
satin    41
twill    39
Name: count, dtype: int64
Train IDs: 78 | Val IDs: 26 | Test IDs: 26
Splits successfully serialized.


In [3]:
# The mechanistic justification for this step is: extract mathematically rigorous texture descriptors (GLCM, LBP, Gabor filter banks, and IFV) characterizing spatial periodicities and intensity distributions.
def extract_specimen_features(img_gray):
    # 1. GLCM features across 4 angles (0, 45, 90, 135 deg) and 4 pixel displacements (1, 2, 4, 8)
    glcm = graycomatrix(img_gray, distances=[1, 2, 4, 8], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], levels=256, symmetric=True, normed=True)
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']
    glcm_feats = [graycoprops(glcm, p).flatten() for p in props]
    glcm_vec = np.concatenate(glcm_feats)

    # 2. Local Binary Patterns (uniform histograms at 3 scales)
    lbp_feats = []
    for p, r in [(8, 1), (16, 2), (24, 3)]:
        lbp = local_binary_pattern(img_gray, p, r, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=p + 2, range=(0, p + 2), density=True)
        lbp_feats.append(hist)
    lbp_vec = np.concatenate(lbp_feats)

    # 3. Gabor Filter Bank (4 radial frequencies x 8 directional orientations = 32 filters)
    gabor_feats = []
    for freq in [0.05, 0.1, 0.2, 0.35]:
        for theta in np.linspace(0, np.pi, 8, endpoint=False):
            kernel = cv2.getGaborKernel((21, 21), 4.0, theta, 1.0/freq, 0.5, 0, ktype=cv2.CV_32F)
            filtered = cv2.filter2D(img_gray.astype(np.float32), cv2.CV_32F, kernel)
            gabor_feats.extend([float(np.mean(filtered)), float(np.std(filtered)), float(np.mean(filtered**2))])
    gabor_vec = np.array(gabor_feats)

    # 4. Intensity Feature Vector (IFV) & 1D directional projections
    h_proj = np.mean(img_gray, axis=1)
    v_proj = np.mean(img_gray, axis=0)
    ifv_vec = np.array([
        float(np.mean(img_gray)), float(np.var(img_gray)), float(np.std(img_gray)),
        float(stats.skew(img_gray.ravel())), float(stats.kurtosis(img_gray.ravel())),
        float(np.percentile(img_gray, 10)), float(np.percentile(img_gray, 25)),
        float(np.percentile(img_gray, 50)), float(np.percentile(img_gray, 75)),
        float(np.percentile(img_gray, 90)),
        float(np.var(h_proj)), float(np.std(h_proj)),
        float(np.var(v_proj)), float(np.std(v_proj)),
        float(stats.skew(h_proj)), float(stats.skew(v_proj))
    ])

    return np.concatenate([glcm_vec, lbp_vec, gabor_vec, ifv_vec])

features_cache = "experiments/learned_models/results/fabricnet_features.npz"
if os.path.exists(features_cache):
    cached = np.load(features_cache)
    X_all = cached['X']
    ids_all = cached['ids']
    print(f"Loaded {len(ids_all)} cached feature vectors from {features_cache}.")
else:
    print("Extracting features for all 130 specimens...")
    t0 = time.time()
    feature_list = []
    id_list = []
    for idx, row in df_meta.iterrows():
        img_id = int(row['Image id'])
        img_path = f"data/fabricnet/W{img_id:03d}.jpg"
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        feat = extract_specimen_features(img)
        feature_list.append(feat)
        id_list.append(img_id)
    X_all = np.array(feature_list)
    ids_all = np.array(id_list)
    np.savez_compressed(features_cache, X=X_all, ids=ids_all)
    print(f"Extracted {len(ids_all)} feature vectors in {time.time() - t0:.2f}s. Saved to {features_cache}.")

print(f"Feature matrix shape: {X_all.shape}")


Extracting features for all 130 specimens...


Extracted 130 feature vectors in 80.87s. Saved to experiments/learned_models/results/fabricnet_features.npz.
Feature matrix shape: (130, 262)


In [4]:
# The mechanistic justification for this step is: align specimen feature matrices with targets and scale input distributions using statistics calculated exclusively on the training partition.
id_to_idx = {int(img_id): i for i, img_id in enumerate(ids_all)}

train_indices = [id_to_idx[i] for i in train_ids]
val_indices = [id_to_idx[i] for i in val_ids]
test_indices = [id_to_idx[i] for i in test_ids]

X_train = X_all[train_indices]
X_val = X_all[val_indices]
X_test = X_all[test_indices]

# Target extraction
meta_by_id = df_meta.set_index('Image id')

y_train_warp = meta_by_id.loc[train_ids, 'Warp'].values.astype(np.float32)
y_train_weft = meta_by_id.loc[train_ids, 'Weft'].values.astype(np.float32)

y_val_warp = meta_by_id.loc[val_ids, 'Warp'].values.astype(np.float32)
y_val_weft = meta_by_id.loc[val_ids, 'Weft'].values.astype(np.float32)

y_test_warp = meta_by_id.loc[test_ids, 'Warp'].values.astype(np.float32)
y_test_weft = meta_by_id.loc[test_ids, 'Weft'].values.astype(np.float32)
test_textures = meta_by_id.loc[test_ids, 'Texture'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("X_val shape:", X_val_scaled.shape)
print("X_test shape:", X_test_scaled.shape)


X_train shape: (78, 262)
X_val shape: (26, 262)
X_test shape: (26, 262)


In [5]:
# The mechanistic justification for this step is: fit three distinct functional approximation paradigms (gradient boosting, bagged decision trees, multi-layer perceptron) on texture features to predict warp and weft densities.
def fit_and_evaluate(name, model_warp, model_weft, use_scaled=True):
    X_tr = X_train_scaled if use_scaled else X_train
    X_v = X_val_scaled if use_scaled else X_val
    X_te = X_test_scaled if use_scaled else X_test

    # Fit warp
    model_warp.fit(X_tr, y_train_warp)
    pred_test_warp = model_warp.predict(X_te)

    # Fit weft
    model_weft.fit(X_tr, y_train_weft)
    pred_test_weft = model_weft.predict(X_te)

    mae_warp = mean_absolute_error(y_test_warp, pred_test_warp)
    rmse_warp = np.sqrt(mean_squared_error(y_test_warp, pred_test_warp))
    r2_warp = r2_score(y_test_warp, pred_test_warp)

    mae_weft = mean_absolute_error(y_test_weft, pred_test_weft)
    rmse_weft = np.sqrt(mean_squared_error(y_test_weft, pred_test_weft))
    r2_weft = r2_score(y_test_weft, pred_test_weft)

    return {
        "model": name,
        "pred_warp": pred_test_warp,
        "pred_weft": pred_test_weft,
        "warp": {"mae": float(mae_warp), "rmse": float(rmse_warp), "r2": float(r2_warp)},
        "weft": {"mae": float(mae_weft), "rmse": float(rmse_weft), "r2": float(r2_weft)}
    }

# 1. XGBoost
xgb_warp = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED)
xgb_weft = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=SEED)
res_xgb = fit_and_evaluate("XGBoost", xgb_warp, xgb_weft, use_scaled=False)

# 2. Random Forest
rf_warp = RandomForestRegressor(n_estimators=150, max_depth=8, min_samples_split=3, random_state=SEED)
rf_weft = RandomForestRegressor(n_estimators=150, max_depth=8, min_samples_split=3, random_state=SEED)
res_rf = fit_and_evaluate("RandomForest", rf_warp, rf_weft, use_scaled=False)

# 3. Multi-Layer Perceptron (MLP)
mlp_warp = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=600, alpha=0.01, random_state=SEED, early_stopping=True)
mlp_weft = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', max_iter=600, alpha=0.01, random_state=SEED, early_stopping=True)
res_mlp = fit_and_evaluate("MLPRegressor", mlp_warp, mlp_weft, use_scaled=True)

models_results = [res_xgb, res_rf, res_mlp]

print(f"{'Model':<15} | {'Warp MAE':<9} {'Warp RMSE':<10} {'Warp R2':<8} | {'Weft MAE':<9} {'Weft RMSE':<10} {'Weft R2':<8}")
print("-" * 75)
for r in models_results:
    print(f"{r['model']:<15} | {r['warp']['mae']:<9.3f} {r['warp']['rmse']:<10.3f} {r['warp']['r2']:<8.3f} | {r['weft']['mae']:<9.3f} {r['weft']['rmse']:<10.3f} {r['weft']['r2']:<8.3f}")


Model           | Warp MAE  Warp RMSE  Warp R2  | Weft MAE  Weft RMSE  Weft R2 
---------------------------------------------------------------------------
XGBoost         | 4.830     7.453      0.702    | 6.599     12.282     0.339   
RandomForest    | 4.882     7.600      0.690    | 5.389     6.970      0.787   
MLPRegressor    | 7.114     8.993      0.566    | 13.540    16.986     -0.265  


In [6]:
# The mechanistic justification for this step is: extract deterministic engine predictions for the exact 26 held-out test specimens and benchmark side-by-side per weave type.
with open("results/fabricnet_benchmark.json") as f:
    bench_data = json.load(f)

det_by_num = {s['sample_num']: s for s in bench_data['samples']}

det_warp_preds = np.array([det_by_num[i]['pred_warp'] for i in test_ids])
det_weft_preds = np.array([det_by_num[i]['pred_weft'] for i in test_ids])

det_mae_warp = float(mean_absolute_error(y_test_warp, det_warp_preds))
det_rmse_warp = float(np.sqrt(mean_squared_error(y_test_warp, det_warp_preds)))
det_r2_warp = float(r2_score(y_test_warp, det_warp_preds))

det_mae_weft = float(mean_absolute_error(y_test_weft, det_weft_preds))
det_rmse_weft = float(np.sqrt(mean_squared_error(y_test_weft, det_weft_preds)))
det_r2_weft = float(r2_score(y_test_weft, det_weft_preds))

res_det = {
    "model": "Deterministic Engine",
    "pred_warp": det_warp_preds,
    "pred_weft": det_weft_preds,
    "warp": {"mae": det_mae_warp, "rmse": det_rmse_warp, "r2": det_r2_warp},
    "weft": {"mae": det_mae_weft, "rmse": det_rmse_weft, "r2": det_r2_weft}
}

all_evals = models_results + [res_det]

print("\\n=== COMPLETE COMPARISON ON 26 HELD-OUT TEST SPECIMENS ===")
print(f"{'Method':<22} | {'Warp MAE':<9} {'Warp RMSE':<10} {'Warp R2':<8} | {'Weft MAE':<9} {'Weft RMSE':<10} {'Weft R2':<8}")
print("-" * 85)
for r in all_evals:
    print(f"{r['model']:<22} | {r['warp']['mae']:<9.3f} {r['warp']['rmse']:<10.3f} {r['warp']['r2']:<8.3f} | {r['weft']['mae']:<9.3f} {r['weft']['rmse']:<10.3f} {r['weft']['r2']:<8.3f}")

# Breakdown per weave architecture
unique_textures = np.unique(test_textures)
weave_breakdown = {}
print("\\n=== PER-WEAVE BREAKDOWN (TEST SET) ===")
for tex in unique_textures:
    mask = (test_textures == tex)
    count = int(np.sum(mask))
    print(f"\\n--- Texture: {tex.upper()} (N={count}) ---")
    print(f"{'Method':<22} | {'Warp MAE':<9} {'Weft MAE':<9}")
    weave_breakdown[tex] = {}
    for r in all_evals:
        m_w = float(mean_absolute_error(y_test_warp[mask], r['pred_warp'][mask]))
        m_wf = float(mean_absolute_error(y_test_weft[mask], r['pred_weft'][mask]))
        weave_breakdown[tex][r['model']] = {"warp_mae": m_w, "weft_mae": m_wf}
        print(f"{r['model']:<22} | {m_w:<9.3f} {m_wf:<9.3f}")

# Serialize complete results
output_summary = {
    "test_specimen_count": len(test_ids),
    "test_ids": test_ids,
    "overall_metrics": {r['model']: {"warp": r['warp'], "weft": r['weft']} for r in all_evals},
    "per_weave_breakdown": weave_breakdown
}

with open("experiments/learned_models/results/fabricnet_regression_results.json", "w") as f:
    json.dump(output_summary, f, indent=2)

print("\\nResults successfully saved to experiments/learned_models/results/fabricnet_regression_results.json")


\n=== COMPLETE COMPARISON ON 26 HELD-OUT TEST SPECIMENS ===
Method                 | Warp MAE  Warp RMSE  Warp R2  | Weft MAE  Weft RMSE  Weft R2 
-------------------------------------------------------------------------------------
XGBoost                | 4.830     7.453      0.702    | 6.599     12.282     0.339   
RandomForest           | 4.882     7.600      0.690    | 5.389     6.970      0.787   
MLPRegressor           | 7.114     8.993      0.566    | 13.540    16.986     -0.265  
Deterministic Engine   | 13.271    18.630     -0.861   | 13.162    17.846     -0.397  
\n=== PER-WEAVE BREAKDOWN (TEST SET) ===
\n--- Texture: PLAIN (N=10) ---
Method                 | Warp MAE  Weft MAE 
XGBoost                | 3.699     4.618    
RandomForest           | 3.367     4.750    
MLPRegressor           | 5.875     10.671   
Deterministic Engine   | 8.981     8.048    
\n--- Texture: SATIN (N=8) ---
Method                 | Warp MAE  Weft MAE 
XGBoost                | 3.564     8.112    
